Retrieve research topics from OpenAlex for all works that cite the original pymatgen paper.
Then use a LLM to organize these topics into a mindmap.

Note:
- You would need to set `OPENAI_API_KEY`.

In [ ]:
# Step 1: Get topics from OpenAlex

from collections import Counter

import requests

url = "https://api.openalex.org/works"
params = {"filter": "cites:w2015197254", "group_by": "primary_topic.id", "per_page": 50}
response = requests.get(url, params=params, timeout=10)
data = response.json()

topic_counter = Counter(
    {
        group.get("key_display_name", "None"): group["count"]
        for group in data.get("group_by")
        if group["count"] > 1  # Top 50 is around 10
    }
)

print("Topics from OpenAlex:")
for topic, count in topic_counter.items():
    print(f"{topic:<60}{count}")

Topics from OpenAlex:
Machine Learning in Materials Science                       1144
Advanced Battery Materials and Technologies                 323
Advancements in Battery Materials                           322
Perovskite Materials and Applications                       107
Electrocatalysts for Energy Conversion                      75
Metal-Organic Frameworks: Synthesis and Applications        74
Electronic and Structural Properties of Oxides              67
2D Materials and Applications                               66
Advanced Thermoelectric Materials and Devices               55
MXene and MAX Phase Materials                               49
Synthesis and Properties of Inorganic Cluster Compounds     41
X-ray Diffraction in Crystallography                        37
Chalcogenide Semiconductor Thin Films                       35
Advancements in Solid Oxide Fuel Cells                      31
High Entropy Alloys Studies                                 31
Ammonia Synthesis and Nitrog

In [ ]:
# Step 2: Sse LLM to group topics

from openai import OpenAI

client = OpenAI()  # https://platform.openai.com/docs/overview

topic_list = "\n".join(f"- {topic} ({count})" for topic, count in topic_counter.items())

prompt = f"""
The following research topics are derived from papers that cite the pymatgen library, with each number indicating the count of citing works in that area.

Please organize them into 3-6 thematic branches that represent broader areas of materials research (e.g., batteries, machine learning, catalysis, ...). Return  output in a plain text tree structure (with indentation).

Topics:
{topic_list}
"""

chat_response = client.responses.create(
    model="gpt-4.1",
    input=prompt,
)

print("\nMind Map Summary:\n")
print(chat_response.output_text)


Mind Map Summary:

Here’s a thematic grouping in plain text tree format, organizing the topics into broader branches of materials research:

```
Energy Storage and Conversion
    Advanced Battery Materials and Technologies (323)
    Advancements in Battery Materials (322)
    Advanced battery technologies research (17)
    Hydrogen Storage and Materials (18)
    Advancements in Solid Oxide Fuel Cells (31)
    Advanced Thermoelectric Materials and Devices (55)
    Electrocatalysts for Energy Conversion (75)
    CO2 Reduction Techniques and Catalysts (19)
    Ammonia Synthesis and Nitrogen Reduction (27)
    Chemical Looping and Thermochemical Processes (13)

Catalysis and Functional Materials
    Catalytic Processes in Materials Science (25)
    Zeolite Catalysis and Synthesis (10)
    Catalysis and Oxidation Reactions (13)
    Advanced Photocatalysis Techniques (21)
    Metal-Organic Frameworks: Synthesis and Applications (74)
    Perovskite Materials and Applications (107)
    Magnet